## Silver Layer

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# =========================================================
# Read Bronze incrementally
# =========================================================
df_bronze = (
    spark.readStream.table("fintech.bronze.stock_prices")
)

def merge_to_silver(batch_df, batch_id):

    # =====================================================
    # Data Quality
    # =====================================================

    df_validated = (
        batch_df
        .filter(col("symbol").isNotNull())
        .filter(col("trade_date").isNotNull())
        .filter(col("open") > 0)
        .filter(col("high") > 0)
        .filter(col("low") > 0)
        .filter(col("close") > 0)
        .filter(col("volume") > 0)
        .filter(col("high") >= col("open"))
        .filter(col("high") >= col("close"))
        .filter(col("low") <= col("open"))
        .filter(col("low") <= col("close"))
    )

    # =====================================================
    # Deduplication
    # =====================================================

    window = (
        Window
        .partitionBy(
            "symbol",
            "trade_date"
        )
        .orderBy(
            col("_ingestion_timestamp").desc()
        )
    )

    df_silver = (
        df_validated
        .withColumn(
            "_row_number",
            row_number().over(window)
        )
        .filter(
            col("_row_number") == 1
        )
        .drop("_row_number")
    )

    # =====================================================
    # MERGE
    # =====================================================

    silver_table = DeltaTable.forName(
        spark,
        "fintech.silver.stock_prices"
    )

    (
        silver_table.alias("target")
        .merge(
            df_silver.alias("source"),
            """
            target.symbol = source.symbol
            AND target.trade_date = source.trade_date
            """
        )
        .whenMatchedUpdate(
            condition="""
                source._ingestion_timestamp
                > target._ingestion_timestamp
            """,
            set={
                "open": "source.open",
                "high": "source.high",
                "low": "source.low",
                "close": "source.close",
                "volume": "source.volume",
                "change": "source.change",
                "change_percent": "source.change_percent",
                "vwap": "source.vwap",
                "_ingestion_timestamp": "source._ingestion_timestamp",
                "_source_file": "source._source_file"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

# =========================================================
# Start streaming
# =========================================================
query = (
    df_bronze.writeStream
    .foreachBatch(merge_to_silver)
    .option(
        "checkpointLocation", 
        "/Volumes/fintech/silver/checkpoints/silver_stock_prices/"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()